In [ ]:
# ============================================================
# CELL T1 — Install Training Dependencies (Fixed)
# ============================================================

!pip install tensorflow
!pip install openwakeword
!pip install librosa soundfile
!pip install tf2onnx
!sudo apt-get install -y ffmpeg

# Verify installations
import tensorflow as tf
import openwakeword
import librosa
import tf2onnx
import numpy as np
import os

print("All packages installed successfully.")
print(f"TensorFlow version : {tf.__version__}")
print(f"tf2onnx version    : {tf2onnx.__version__}")
print(f"librosa version    : {librosa.__version__}")
print("Ready for Cell T2.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
All packages installed successfully.
TensorFlow version : 2.19.0
tf2onnx version    : 1.17.0
librosa version    : 0.11.0
Ready for Cell T2.


In [ ]:
# ============================================================
# REMOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
import os

# Force remount
drive.mount('/content/drive', force_remount=True)

# Verify folders exist
POSITIVE_DIR = "/content/drive/MyDrive/wake_word/positive"
NEGATIVE_DIR = "/content/drive/MyDrive/wake_word/negative"

print(f"Positive folder exists: {os.path.exists(POSITIVE_DIR)}")
print(f"Negative folder exists: {os.path.exists(NEGATIVE_DIR)}")


Mounted at /content/drive
Positive folder exists: True
Negative folder exists: True


In [ ]:
# ============================================================
# DIAGNOSTIC CELL — Check folder contents
# ============================================================

import os

POSITIVE_DIR = "/content/drive/MyDrive/wake_word/positive"
NEGATIVE_DIR = "/content/drive/MyDrive/wake_word/negative"

# Check if folders exist
print("Checking folders...")
print(f"Positive folder exists: {os.path.exists(POSITIVE_DIR)}")
print(f"Negative folder exists: {os.path.exists(NEGATIVE_DIR)}")

# List ALL files in positive folder
print("\nFiles in POSITIVE folder:")
if os.path.exists(POSITIVE_DIR):
    all_files = os.listdir(POSITIVE_DIR)
    if len(all_files) == 0:
        print("   EMPTY — no files found!")
    for f in all_files:
        print(f"   '{f}'")

# List ALL files in negative folder
print("\nFiles in NEGATIVE folder:")
if os.path.exists(NEGATIVE_DIR):
    all_files = os.listdir(NEGATIVE_DIR)
    if len(all_files) == 0:
        print("   EMPTY — no files found!")
    for f in all_files:
        print(f"   '{f}'")

# Also check parent folder
print("\nContents of /content/drive/MyDrive/wake_word/:")
parent = "/content/drive/MyDrive/wake_word"
if os.path.exists(parent):
    for f in os.listdir(parent):
        print(f"   '{f}'")
else:
    print("   wake_word folder not found!")

Checking folders...
Positive folder exists: True
Negative folder exists: True

Files in POSITIVE folder:
   'hey_suji_01.wav.m4a'
   'hey_suji_09.wav.m4a'
   'hey_suji_06.wav.m4a'
   'hey_suji_02.wav.m4a'
   'hey_suji_04.wav.m4a'
   'hey_suji_07.wav.m4a'
   'hey_suji_05.wav.m4a'
   'hey_suji_10.wav.m4a'
   'hey_suji_03.wav.m4a'
   'hey_suji_08.wav.m4a'
   'hey_suji_14.wav.m4a'
   'hey_suji_11.wav.m4a'
   'hey_suji_12.wav.m4a'
   'hey_suji_13.wav.m4a'
   'hey_suji_15.wav.m4a'
   'hey_suji_16.wav.m4a'
   'hey_suji_19.wav.m4a'
   'hey_suji_18.wav.m4a'
   'hey_suji_17.wav.m4a'
   'hey_suji_20.wav.m4a'
   'hey_suji_24.wav.m4a'
   'hey_suji_26.wav.m4a'
   'hey_suji_22.wav.m4a'
   'hey_suji_23.wav.m4a'
   'hey_suji_28.wav.m4a'
   'hey_suji_21.wav.m4a'
   'hey_suji_27.wav.m4a'
   'hey_suji_25.wav.m4a'
   'hey_suji_29.wav.m4a'
   'hey_suji_30.wav.m4a'

Files in NEGATIVE folder:
   'negative_02.wav.m4a'
   'negative_10.wav.m4a'
   'negative_05.wav.m4a'
   'negative_09.wav.m4a'
   'negative_08.wa

In [ ]:
# ============================================================
# CELL T2 — Fixed for .wav.m4a files
# ============================================================

import numpy as np
import librosa
import os
import soundfile as sf
from google.colab import drive

# Drive already mounted, skip remounting
# drive.mount('/content/drive')

POSITIVE_DIR  = "/content/drive/MyDrive/wake_word/positive"
NEGATIVE_DIR  = "/content/drive/MyDrive/wake_word/negative"
OUTPUT_DIR    = "/content/wake_word_processed"
SAMPLE_RATE   = 16000
CLIP_DURATION = 1.5
N_MELS        = 40
HOP_LENGTH    = 160
N_FFT         = 480

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# Fix — accept any audio extension
# ─────────────────────────────────────────────

ACCEPTED_EXTENSIONS = ('.wav', '.m4a', '.wav.m4a', '.mp3', '.ogg')

def is_audio_file(filename: str) -> bool:
    """Returns True if file is any supported audio format."""
    return any(filename.endswith(ext) for ext in ACCEPTED_EXTENSIONS)


def load_and_preprocess(file_path: str) -> np.ndarray:
    """
    Loads any audio file, converts to 16kHz mono,
    clips/pads to fixed duration.
    """
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
        target_length = int(SAMPLE_RATE * CLIP_DURATION)

        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]

        return audio

    except Exception as e:
        print(f"Failed to load {file_path}: {e}")
        return None


def extract_features(audio: np.ndarray) -> np.ndarray:
    """Converts raw audio to mel spectrogram features."""
    mel_spec = librosa.feature.melspectrogram(
        y          = audio,
        sr         = SAMPLE_RATE,
        n_mels     = N_MELS,
        hop_length = HOP_LENGTH,
        n_fft      = N_FFT
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-9)
    return log_mel.T


def augment_audio(audio: np.ndarray) -> list:
    """Creates variations of a single recording."""
    augmented = [audio]

    # Add slight noise
    noise = np.random.randn(len(audio)) * 0.005
    augmented.append(audio + noise)

    # Louder
    augmented.append(audio * 1.3)

    # Quieter
    augmented.append(audio * 0.7)

    # Pitch up
    try:
        augmented.append(librosa.effects.pitch_shift(
            audio, sr=SAMPLE_RATE, n_steps=1))
    except:
        pass

    # Pitch down
    try:
        augmented.append(librosa.effects.pitch_shift(
            audio, sr=SAMPLE_RATE, n_steps=-1))
    except:
        pass

    return augmented


def load_dataset(positive_dir: str, negative_dir: str):
    """Loads all samples and prepares X, y arrays."""
    X = []
    y = []

    # --- Positive samples ---
    print("Loading positive samples (Hey Suji)...")
    pos_files = sorted([f for f in os.listdir(positive_dir)
                        if is_audio_file(f)])

    for fname in pos_files:
        fpath = os.path.join(positive_dir, fname)
        audio = load_and_preprocess(fpath)
        if audio is None:
            continue
        variations = augment_audio(audio)
        for var in variations:
            features = extract_features(var)
            X.append(features)
            y.append(1)
        print(f"   Loaded + augmented: {fname} ({len(variations)} variations)")

    # --- Negative samples ---
    print("\nLoading negative samples (not Hey Suji)...")
    neg_files = sorted([f for f in os.listdir(negative_dir)
                        if is_audio_file(f)])

    for fname in neg_files:
        fpath = os.path.join(negative_dir, fname)
        audio = load_and_preprocess(fpath)
        if audio is None:
            continue
        features = extract_features(audio)
        X.append(features)
        y.append(0)
        print(f"   Loaded: {fname}")

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    return X, y


# ─────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────

print("Starting dataset preparation...")
X, y = load_dataset(POSITIVE_DIR, NEGATIVE_DIR)

n_positive = int(np.sum(y == 1))
n_negative = int(np.sum(y == 0))

print("\n--- Dataset Summary ---")
print(f"Total samples       : {len(X)}")
print(f"Positive (Hey Suji) : {n_positive}")
print(f"Negative (other)    : {n_negative}")
print(f"Feature shape       : {X.shape}")
print(f"Label shape         : {y.shape}")
print("\nReady for Cell T3 - Training!")

Starting dataset preparation...
Loading positive samples (Hey Suji)...


/tmp/ipykernel_16631/1915977809.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


   Loaded + augmented: hey_suji_01.wav.m4a (6 variations)


/tmp/ipykernel_16631/1915977809.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


   Loaded + augmented: hey_suji_02.wav.m4a (6 variations)


...

/tmp/ipykernel_16631/1915977809.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


   Loaded: negative_28.wav.m4a


/tmp/ipykernel_16631/1915977809.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


   Loaded: negative_29.wav.m4a
   Loaded: negative_30.wav.m4a

--- Dataset Summary ---
Total samples       : 210
Positive (Hey Suji) : 180
Negative (other)    : 30
Feature shape       : (210, 151, 40)
Label shape         : (210,)

Ready for Cell T3 - Training!


/tmp/ipykernel_16631/1915977809.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
# ============================================================
# CELL T3 FIX 2 — Convert trained model to ONNX
# ============================================================

import tf2onnx
import tensorflow as tf
import numpy as np
import os
import shutil

# Load the best saved model
print("Loading best saved model...")
model = tf.keras.models.load_model('/content/hey_suji.keras')
print("Model loaded successfully.")

print("Converting to ONNX format...")

# Get input shape from model
input_shape = model.input_shape[1:]  # (151, 40, 1)
print(f"Input shape: {input_shape}")

# Define input signature separately
input_signature = [tf.TensorSpec(
    shape = [None] + list(input_shape),
    dtype = tf.float32,
    name  = 'input'
)]

# Create concrete function using input signature
@tf.function
def model_fn(x):
    return model(x, training=False)

concrete_func = model_fn.get_concrete_function(
    tf.TensorSpec(shape=[None] + list(input_shape), dtype=tf.float32)
)

# Convert using from_function with input_signature passed explicitly
model_proto, _ = tf2onnx.convert.from_function(
    function         = model_fn,
    input_signature  = input_signature,
    opset            = 13,
    output_path      = "/content/hey_suji.onnx"
)

print("Model converted to ONNX successfully.")

# Verify file
size_kb = os.path.getsize("/content/hey_suji.onnx") / 1024
print(f"ONNX file size : {size_kb:.1f} KB")
print(f"Saved to       : /content/hey_suji.onnx")

# Save to Google Drive
drive_save_path = "/content/drive/MyDrive/wake_word/hey_suji.onnx"
shutil.copy("/content/hey_suji.onnx", drive_save_path)
print(f"Backup saved to Google Drive: {drive_save_path}")
print("\nTraining complete! Use hey_suji.onnx in your main pipeline.")

Loading best saved model...
Model loaded successfully.
Converting to ONNX format...
Input shape: (151, 40, 1)
Model converted to ONNX successfully.
ONNX file size : 113.5 KB
Saved to       : /content/hey_suji.onnx
Backup saved to Google Drive: /content/drive/MyDrive/wake_word/hey_suji.onnx

Training complete! Use hey_suji.onnx in your main pipeline.
